# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from catboost import CatBoostClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [5]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510,0.826590
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514,0.801335
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012,0.782757
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494,0.502476
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416,0.703165


In [6]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.019849,0.509303
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.012173,0.505573
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.010304,0.504827
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.444558,0.696136
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.939648,0.852833


# Machine Learning

In [7]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 14:58:23,093] A new study created in memory with name: no-name-08c819e9-8a9c-4889-abf8-a79f9842db1b
Best trial: 5. Best value: 0.949618:   0%|          | 1/500 [00:15<2:05:42, 15.12s/it]

[I 2026-05-18 14:58:38,206] Trial 5 finished with value: 0.9496179463819487 and parameters: {'solver': 'saga', 'C': 0.000571582059976165, 'l1_ratio': 0.7144304416479819, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.0011410774867599521, 'max_iter': 2003}. Best is trial 5 with value: 0.9496179463819487.
[I 2026-05-18 14:58:38,217] Trial 6 finished with value: 0.9494281854547953 and parameters: {'solver': 'saga', 'C': 0.19816457741260096, 'l1_ratio': 0.032121095903444696, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0005588455800719528, 'max_iter': 4360}. Best is trial 5 with value: 0.9496179463819487.


Best trial: 5. Best value: 0.949618:   1%|          | 3/500 [00:15<33:56,  4.10s/it]  

[I 2026-05-18 14:58:38,690] Trial 4 finished with value: 0.9492946526973484 and parameters: {'solver': 'saga', 'C': 0.0002936101707099479, 'l1_ratio': 0.22501943435526905, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0005089049028739349, 'max_iter': 3023}. Best is trial 5 with value: 0.9496179463819487.


Best trial: 5. Best value: 0.949618:   1%|          | 4/500 [00:16<24:26,  2.96s/it]

...

[I 2026-05-18 15:12:09,983] Trial 499 finished with value: 0.9497147277499796 and parameters: {'solver': 'saga', 'C': 0.001181422874897922, 'l1_ratio': 0.9719070637595604, 'class_weight': None, 'fit_intercept': False, 'tol': 2.4026630220742177e-05, 'max_iter': 4926}. Best is trial 449 with value: 0.9497176611404201.


Best trial: 449. Best value: 0.949718: 100%|█████████▉| 499/500 [13:47<00:00,  1.16it/s]

[I 2026-05-18 15:12:10,607] Trial 494 finished with value: 0.9497159612755766 and parameters: {'solver': 'saga', 'C': 0.0011274317422515713, 'l1_ratio': 0.970201738692388, 'class_weight': None, 'fit_intercept': False, 'tol': 4.596547579317832e-06, 'max_iter': 4986}. Best is trial 449 with value: 0.9497176611404201.


Best trial: 449. Best value: 0.949718: 100%|██████████| 500/500 [13:47<00:00,  1.66s/it]

[I 2026-05-18 15:12:11,039] Trial 498 finished with value: 0.9497161670492785 and parameters: {'solver': 'saga', 'C': 0.001107002946324631, 'l1_ratio': 0.9723273495761446, 'class_weight': None, 'fit_intercept': False, 'tol': 7.926487046216622e-06, 'max_iter': 4989}. Best is trial 449 with value: 0.9497176611404201.
Best trial score:
0.9497176611404201

Best params:
{'solver': 'saga', 'C': 0.0009284772726825259, 'l1_ratio': 0.9569068811377953, 'class_weight': None, 'fit_intercept': False, 'tol': 0.00023038949336779107, 'max_iter': 4765}


In [8]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_train, y_train.PitNextLap)

# Submission

In [9]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [10]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [11]:
submission.to_csv('../data/submission/submission.csv', index=False)